# Zero-Shot Model Comparison – Chronos & Kronos

Runs **Chronos-bolt-base** and **Kronos-base** in zero-shot mode on energy-sector financial data (test split: 2021+).
Both models are evaluated across 5 seeds; an additional Kronos run uses the optimal inference parameters
identified in the data-parameter sensitivity analysis for a fair comparison against the fine-tuned variant.

| Setting | Value | Source |
|---|---|---|
| Seeds | 13, 42, 123, 456, 789 | Multi-seed robustness |
| Config | `config/energy_assets_filtered.yaml` | 104-Asset-Universum |
| Context | 80 | HANDOFF |
| Forecast | 12 | HANDOFF |

**Workflow:**
1. Setup environment
2. Run Chronos zero-shot (5 seeds)
3. Run Kronos zero-shot (5 seeds)
4. Aggregate per-seed metrics
5. Kronos base run with optimal params (context=120, forecast=6) for fine-tuning comparison

## Setup: Clone Repository

In [ ]:
!rm -rf /content/BA
print("Cleaned up.")

In [ ]:
import os
from pathlib import Path

repo_url = "https://github.com/bp571/BA"

%cd /content
!git clone {repo_url}
%cd /content/BA

TIINGO_API_KEY = "312c6dab6a1fe6258bbc6652bcdec49a14ee08ad"
os.environ["TIINGO_API_KEY"] = TIINGO_API_KEY
Path(".env").write_text(f"TIINGO_API_KEY={TIINGO_API_KEY}\n")
print("✅ API key configured")

# Kronos source repo (gitignored im Hauptrepo)
!git clone https://github.com/shiyu-coder/Kronos.git 02_finetuning/models/Kronos
print("✅ Kronos cloned")

## Install Dependencies

In [ ]:
!pip install -q torch transformers peft gluonts pandas numpy tqdm PyYAML \
    chronos-forecasting einops huggingface_hub yfinance scipy python-dotenv tiingo

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Configuration

Zero-Shot Parameter (HANDOFF: context=80, forecast=12, stride=12) über 5 Seeds auf dem 104-Asset-Universum.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Re-Run auf 104-Asset-Universum (energy_assets_filtered.yaml)
SEEDS    = [13, 42, 123, 456, 789]
CONFIG   = "config/energy_assets_filtered.yaml"

# Zero-Shot Parameter (HANDOFF: c=80, f=12, s=12)
CONTEXT  = 80
FORECAST = 12

print(f"Seeds:    {SEEDS}")
print(f"Config:   {CONFIG}")
print(f"Context:  {CONTEXT}, Forecast: {FORECAST}")

## Chronos Zero-Shot

Runs Chronos-bolt-base across all 5 seeds. Results are written to `01_model_comparison/results/chronos/seed_<seed>/`.

In [ ]:
for seed in SEEDS:
    print(f"\n{'='*60}\nChronos | Seed {seed}\n{'='*60}")
    !python 01_model_comparison/zeroshot/main_chronos.py \
        --seed {seed} \
        --config {CONFIG}

## Kronos Zero-Shot

Runs Kronos-base across all 5 seeds. Results are written to `01_model_comparison/results/kronos/seed_<seed>/`.

In [ ]:
for seed in SEEDS:
    print(f"\n{'='*60}\nKronos | Seed {seed}\n{'='*60}")
    !python 01_model_comparison/zeroshot/main_kronos.py \
        --seed {seed} \
        --config {CONFIG} \
        --context {CONTEXT} \
        --forecast {FORECAST}

## Summary

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

def aggregate(model: str, seed: int) -> dict | None:
    p = Path(f"01_model_comparison/results/{model}/seed_{seed}/final_energy_study.json")
    if not p.exists():
        return None
    with open(p) as f:
        data = json.load(f)
    summary = data["summary"]
    maes    = [v["MAE_indicative"]         for v in summary.values() if "MAE_indicative"         in v]
    ics     = [v["IC_TimeSeries_Mean"]     for v in summary.values() if "IC_TimeSeries_Mean"     in v]
    rankics = [v["RankIC_TimeSeries_Mean"] for v in summary.values() if "RankIC_TimeSeries_Mean" in v]
    return {
        "model":    model,
        "seed":     seed,
        "n_assets": data["n_assets_processed"],
        "time_s":   data["processing_time_seconds"],
        "MAE":      np.mean(maes)    if maes    else np.nan,
        "IC":       np.mean(ics)     if ics     else np.nan,
        "RankIC":   np.mean(rankics) if rankics else np.nan,
    }

rows = []
for model in ["chronos", "kronos"]:
    for seed in SEEDS:
        row = aggregate(model, seed)
        if row:
            rows.append(row)

df = pd.DataFrame(rows)
print("\nPer-Seed Ergebnisse")
print("="*70)
print(df.round(4).to_string(index=False))

print("\nAggregat (Mittelwert ± Std über Seeds)")
print("="*70)
agg = df.groupby("model").agg(
    n_assets=("n_assets", "mean"),
    MAE_mean=("MAE", "mean"), MAE_std=("MAE", "std"),
    IC_mean=("IC", "mean"),   IC_std=("IC", "std"),
    RankIC_mean=("RankIC", "mean"), RankIC_std=("RankIC", "std"),
    total_time_s=("time_s", "sum"),
).round(4)
print(agg.to_string())

## Kronos Base mit optimalen Parametern (context=120, forecast=6, seed=42)

Fairer Vergleich mit Kronos Fine-tuned — identische Parameter.

In [ ]:
!python 01_model_comparison/zeroshot/main_kronos.py \
    --context 120 \
    --forecast 6 \
    --seed 42 \
    --config config/energy_assets_filtered.yaml

## Vergleich: Kronos Base vs. Kronos Fine-tuned (context=120, forecast=6, seed=42)

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

def load_metrics(path):
    with open(path) as f:
        d = json.load(f)
    summary = d['summary']
    maes    = [v['MAE_indicative']        for v in summary.values() if 'MAE_indicative'        in v]
    ics     = [v['IC_TimeSeries_Mean']    for v in summary.values() if 'IC_TimeSeries_Mean'    in v]
    rankics = [v['RankIC_TimeSeries_Mean'] for v in summary.values() if 'RankIC_TimeSeries_Mean' in v]
    return {
        'MAE':    np.mean(maes),
        'IC':     np.mean(ics),
        'RankIC': np.mean(rankics),
        'RankIC>0': sum(r > 0 for r in rankics),
        'n': len(rankics),
    }

base_path = Path("01_model_comparison/results/kronos/seed_42/final_energy_study.json")
ft_path   = Path("02_finetuning/results/kronos_finetuned/seed_42/final_energy_study.json")

rows = []
if base_path.exists():
    rows.append({'Model': 'Kronos Base', **load_metrics(base_path)})
else:
    print("Kronos Base result not found — run cell above first.")

if ft_path.exists():
    rows.append({'Model': 'Kronos Fine-tuned', **load_metrics(ft_path)})
else:
    print("Kronos Fine-tuned result not found.")

if rows:
    df = pd.DataFrame(rows).set_index('Model')
    df['RankIC>0'] = df.apply(lambda r: f"{int(r['RankIC>0'])}/{int(r['n'])}", axis=1)
    df = df.drop(columns='n')
    df = df.round(4)
    print("\nKronos Base vs. Kronos Fine-tuned  (context=120, forecast=6, seed=42, test: 2021–)")
    print("="*65)
    print(df.to_string())
    print()
    if len(rows) == 2:
        base, ft = rows
        for m in ['MAE', 'IC', 'RankIC']:
            delta = ft[m] - base[m]
            sign  = '+' if delta >= 0 else ''
            better = '↑' if (m == 'MAE' and delta < 0) or (m != 'MAE' and delta > 0) else '↓'
            print(f"  {m:8s}  delta = {sign}{delta:.4f}  {better}")


## Download Results

In [ ]:
from google.colab import files
import zipfile
from pathlib import Path

zip_name = "zeroshot_results.zip"

with zipfile.ZipFile(zip_name, 'w') as zf:
    for model in ["chronos", "kronos"]:
        results_root = Path(f"01_model_comparison/results/{model}")
        if not results_root.exists():
            continue
        for f in results_root.rglob("*.json"):
            zf.write(f, arcname=str(f.relative_to("01_model_comparison/results")))

files.download(zip_name)
print(f"Downloaded: {zip_name}")